# DSE 200 — Day 4 Lab, part 1
## Gradient Boosting

**Chunk 8 · sections 8.1–8.10 match the slides · agent on: explain every cell.**

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while we talk. |
| `🔍 READ` | **Predict before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**Open the table of contents** (☰, left edge). *Runtime → Run all* if you get lost.

In [ ]:
# ▶ SETUP — run first. Loads the weather file (about 20 seconds) and defines check_read().
import os, textwrap, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

def find(name):
    """Return a local path for `name`, downloading it from the course repo if needed."""
    for p in [name, "/content/" + name, "data/" + name, "../" + name]:
        if os.path.exists(p):
            return p
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)
        print("downloaded", name, "from the course repo")
        return name
    except Exception as e:
        raise FileNotFoundError(f"{name} not found -- drag it into the Files pane (folder icon, left)") from e

raw = pd.read_csv(find("minute_weather.csv.gz"), parse_dates=["hpwren_timestamp"])
customers = pd.read_csv(find("customers.csv"))

_READ = {
"8.2": ("No. XGBoost scored 0.813: below the depth-4 tree (0.825) and logistic regression (0.823).",
        "Boosting usually wins on tables. Not on this one.\n"
        "One test set is one roll of the dice: 0.823 and 0.825 are 4 customers apart out of 1,879.\n"
        "Before anyone claims a winner, test more than once. That is cross-validation, next."),
"8.5": ("Don't ship it. On the final year it scores 0.889, and one line of code scores 0.902.",
        "The random split put test minutes BETWEEN training minutes. A minute looks almost exactly\n"
        "like the minute before it, and comes with nearly the same answer, so the model could\n"
        "copy a neighbour instead of forecasting. In real use, tomorrow is never in the training data.\n"
        "Train on the past, test on the future."),
}

def check_read(tag, prediction):
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print(f"{len(raw):,} weather rows, {len(customers):,} customers. check_read() ready.")

# One Loop

*Slides: Gradient Boosting · section 8.1*

## 8.1 Boosting By Hand

**Gradient boosting** builds many small decision trees in sequence. Each tree is trained on what the trees before it still get wrong, and their answers are added together.

`X` is the day of the year. `y` is that day's mean temperature at a San Diego weather station in 2013.

In [ ]:
# ▶ DEMO -- one row per day of 2013: the day's mean temperature
in_2013 = raw[raw["hpwren_timestamp"].dt.year == 2013]
daily = in_2013.groupby(in_2013["hpwren_timestamp"].dt.day_of_year)["air_temp"].mean()

X = daily.index.values.reshape(-1, 1)    # day of the year, as a one-column table
y = daily.values                         # mean temperature that day, °F
plt.scatter(X, y, s=8); plt.xlabel("day of 2013"); plt.ylabel("°F"); plt.show()

In [ ]:
# ▶ DEMO -- the entire algorithm
from sklearn.tree import DecisionTreeRegressor

pred = np.full(len(y), y.mean())          # start: every day gets the average
for i in range(101):
    if i in [0, 1, 5, 20, 100]:
        print(f"trees built {i:>3}   squared error {((y - pred) ** 2).mean():6.1f}")
    errors = y - pred                     # what is still wrong?
    tree = DecisionTreeRegressor(max_depth=1).fit(X, errors)   # one tiny tree
    pred = pred + 0.3 * tree.predict(X)   # add a little of its answer

Each tree asks **one** yes/no question about the day ("is it after day 172?"). The `0.3` means no tree is fully trusted. After 100 one-question trees the error is less than a third of where it started.

In [ ]:
# ▶ DEMO -- the same loop, drawn
pred = np.full(len(y), y.mean())
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for i in range(1, 101):
    tree = DecisionTreeRegressor(max_depth=1).fit(X, y - pred)
    pred = pred + 0.3 * tree.predict(X)
    if i in [1, 5, 20, 100]:
        ax = axes[[1, 5, 20, 100].index(i)]
        ax.scatter(X, y, s=5, color="gray"); ax.plot(X, pred, color="orange", lw=2)
        ax.set_title(f"after {i} trees")
plt.show()

**Why "gradient"?** `y - pred` is the negative gradient of the squared error: the direction to move each prediction to reduce the error fastest. Other losses give other gradients; the loop stays the same. For a yes/no target the trees add up to a **score**, and the S-curve from logistic regression turns that score into a probability.

# XGBoost

*Slides: Gradient Boosting · section 8.2*

## 8.2 Back On The Customer Data

Same table, same cleaning, same split as Day 3. Will this customer come back?

In [ ]:
# ▶ SETUP 8.2 -- Day 3's cleaning and split, so the test set is the same 1,879 customers
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

FEATURES = ["n_orders", "total_spend", "avg_order_value",
            "days_since_last_order", "tenure_days", "n_categories"]
c = customers.dropna(subset=FEATURES)
c = c[(c.total_spend >= 0) & (c.n_orders > 0) & (c.tenure_days <= 3650)].copy()
c["avg_order_value"] = (c.total_spend / c.n_orders).round(2)
X_train, X_test, y_train, y_test = train_test_split(
    c[FEATURES], c["returned"], test_size=0.2, random_state=42, stratify=c["returned"])

last_week = {
    "logistic regression": LogisticRegression(max_iter=1000),
    "decision tree, depth 4": DecisionTreeClassifier(max_depth=4, random_state=42),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=42),
}
print(f'always "no"              {1 - y_test.mean():.3f}')
for name, model in last_week.items():
    print(f"{name:<24} {model.fit(X_train, y_train).score(X_test, y_test):.3f}")

In [ ]:
# 🔍 READ 8.2 -- XGBoost: 300 trees, each scaled by 0.1 (the 0.3 in our loop).
#
#     Last week I said gradient boosting would beat both the tree and the forest.
#     Does it? Write the accuracy you expect. Commit first.

prediction = "???"
check_read("8.2", prediction)

In [ ]:
# ▶ DEMO -- the same fit / score as last week
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42)
xgb.fit(X_train, y_train)
xgb.score(X_test, y_test)

**Cross-validation: five tests instead of one.** Cut the training rows into 5 parts. Hold out each part once and train on the other 4. Report the mean and how much it varies.

In [ ]:
# ▶ DEMO -- about a minute
from sklearn.model_selection import StratifiedKFold, cross_val_score

folds = StratifiedKFold(5, shuffle=True, random_state=42)
models = {**last_week,
          "xgboost": xgb,
          "xgboost, gentler": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=folds)
    print(f"{name:<24} {scores.mean():.3f} ± {scores.std():.3f}")

No clear winner, and the simple models hold up. The depth-4 tree leans almost entirely on two features (days since last order, number of orders), so the signal here is simple. **Boosting usually wins on tabular data. "Usually" is why you compare.**

# A Bigger File

*Slides: Gradient Boosting · section 8.3*

## 8.3 `minute_weather.csv`

One row is one minute at a San Diego weather station, Sep 2011 – Sep 2014: temperature, pressure, wind, rain and humidity. 1,587,257 rows, 129 MB uncompressed.

In [ ]:
# ▶ DEMO -- first checks on a new file: how many rows share each timestamp?
raw["hpwren_timestamp"].value_counts().head(2)

**One second of April 2012 appears 32,523 times.** What happened?

In [ ]:
# ▶ DEMO -- a stuck logger: the rows are copies, and every copy has its own ID
print("repeated IDs:", raw["rowID"].duplicated().sum())
df = raw.drop_duplicates(subset=raw.columns.drop("rowID"))
print(f"{len(df):,} rows left")

32,522 exact copies of one reading. A random split could put the same reading in train **and** test. Last week's check, `set(X_train.index) & set(X_test.index)`, would still be empty: it compares row IDs, not contents.

## 8.4 Before Any Model

**The question:** will humidity be under 25% three hours from now? Call that "dry".
**The decision:** a crew planning a controlled burn this afternoon. Dry air can turn it into a wildfire.
**The costs:** a false alarm cancels a day's work. A miss could burn a hillside. They are not the same size.

In [ ]:
# ▶ DEMO -- the label is a column from the future
df = df.sort_values("hpwren_timestamp").reset_index(drop=True)
df["dry_now"] = (df["relative_humidity"] < 25).astype(int)
df["dry_3h"]  = (df["relative_humidity"].shift(-180) < 25).astype(int)   # 180 minutes AHEAD
df = df.iloc[:-180].copy()            # the last 3 hours have no answer yet
df["dry_3h"].mean()

Only the label may look forward. 26% of minutes are followed by dry air 3 hours later. `dry_now` is not the label; we need it shortly.

# Testing On The Future

*Slides: Gradient Boosting · section 8.5*

## 8.5 Exactly What You Did Last Week

A random 80/20 split, and a large model: 500 trees, depth 12.

In [ ]:
# ▶ DEMO -- a few minutes on Colab. Start it, then read the next cell.
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

sensors = ["air_pressure", "air_temp", "avg_wind_direction", "avg_wind_speed",
           "max_wind_direction", "max_wind_speed", "min_wind_direction",
           "min_wind_speed", "rain_accumulation", "rain_duration", "relative_humidity"]

X_tr, X_te, y_tr, y_te = train_test_split(df[sensors], df["dry_3h"],
    test_size=0.2, random_state=42, stratify=df["dry_3h"])
deep = XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=12, random_state=42)
print('always "no"         ', round(1 - y_te.mean(), 3))
print("500 trees, depth 12 ", round(deep.fit(X_tr, y_tr).score(X_te, y_te), 3))

In [ ]:
# 🔍 READ 8.5 -- a seed, stratified, no row in both halves. Everything you were taught.
#
#     Would you give this model to the controlled-burn crew? Yes or no, and why. Commit.

prediction = "???"
check_read("8.5", prediction)

In [ ]:
# ▶ DEMO -- the test it will actually face: train on two years, test on the final year
past  = df[df["hpwren_timestamp"] <  "2013-09-10"]
year3 = df[df["hpwren_timestamp"] >= "2013-09-10"]

deep.fit(past[sensors], past["dry_3h"])
print("its own training rows", round(deep.score(past[sensors], past["dry_3h"]), 3))
print("the final year       ", round(deep.score(year3[sensors], year3["dry_3h"]), 3))

In [ ]:
# ▶ DEMO -- a baseline nobody wrote down: "whatever it is now, it will be in three hours"
print('always "no"  ', round(1 - year3["dry_3h"].mean(), 3))
print("same as now  ", round((year3["dry_now"] == year3["dry_3h"]).mean(), 3))

**The one-line baseline (0.902) beats the 500-tree model (0.889).** On time series "always no" is too easy a baseline. Ask what a lazy forecaster would say.

"Always no" moved from 0.740 to 0.697 because the final year is drier (30% dry, not 26%). Different test rows, different baseline: only compare scores on the same rows.

**Why the random split cheats:** humidity moves about 0.2 points a minute, and the next minute's `dry_3h` is the same 99% of the time. With test minutes scattered between training minutes, the model can look up a neighbour instead of forecasting.

From here on, whole years: **year 1 trains, year 2 chooses settings, year 3 is the final test.**

In [ ]:
# ▶ DEMO -- three years, three jobs
year1 = df[df["hpwren_timestamp"] < "2012-09-10"]
year2 = df[(df["hpwren_timestamp"] >= "2012-09-10") & (df["hpwren_timestamp"] < "2013-09-10")]
for name, part in [("year 1", year1), ("year 2", year2), ("year 3", year3)]:
    print(f"{name}  {len(part):>7,} rows   {part['dry_3h'].mean():.1%} dry in 3 h")

# Settings And Stopping

*Slides: Gradient Boosting · section 8.6*

## 8.6 Three Settings

| Setting | Means | Typical |
| --- | --- | --- |
| `n_estimators` | how many trees | 100 – 3000 |
| `learning_rate` | how much each tree is trusted | 0.01 – 0.3 |
| `max_depth` | how many questions each tree may ask in a row | 3 – 8 |

In [ ]:
# ▶ DEMO -- max_depth: trained on year 1, checked on year 2. Year 3 stays untouched. (2-3 minutes)
for depth in [2, 3, 6, 9, 12]:
    m = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=depth, random_state=42)
    m.fit(year1[sensors], year1["dry_3h"])
    print(f"depth {depth:>2}   year 1 {m.score(year1[sensors], year1['dry_3h']):.3f}"
          f"   year 2 {m.score(year2[sensors], year2['dry_3h']):.3f}")

Deeper trees score better on year 1 and worse on year 2. That widening gap is overfitting. **In boosting, small trees win.**

A lower `learning_rate` needs more trees and ends slightly better. So the real question is not "how many trees?" It is **"when do I stop?"**

In [ ]:
# ▶ DEMO -- early stopping: stop when year 2 hasn't improved for 20 trees
model = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                      early_stopping_rounds=20, random_state=42)
model.fit(year1[sensors], year1["dry_3h"],
          eval_set=[(year2[sensors], year2["dry_3h"])], verbose=False)
model.best_iteration

Asked for 3,000 trees; the best was tree **123**.

**Two traps.** Early stopping needs a **third** split: if you stop by watching the test set, the test set helped build the model. And your agent may write the old form, `fit(..., early_stopping_rounds=20)`, which raises a `TypeError` in current XGBoost; it belongs in the constructor.

# Beating The Baseline

*Slides: Gradient Boosting · section 8.7*

## 8.7 The Same Accuracy, Wrong In Different Ways

In [ ]:
# ▶ DEMO -- count the two mistakes separately
def report(name, pred, truth):
    missed = ((pred == 0) & (truth == 1)).sum()        # dry air we didn't warn about
    alarms = ((pred == 1) & (truth == 0)).sum()        # warnings for nothing
    print(f"{name:<22} accuracy {(pred == truth).mean():.3f}   missed {missed:>6,}   false alarms {alarms:>6,}")

report("same as now", year3["dry_now"], year3["dry_3h"])
report("xgboost, sensors", model.predict(year3[sensors]), year3["dry_3h"])

The model sees only this minute's readings. What is it missing? **How things are changing.**

In [ ]:
# ▶ DEMO -- features from the past (feature engineering, now on a time series)
df["hour"]  = df["hpwren_timestamp"].dt.hour
df["month"] = df["hpwren_timestamp"].dt.month
df["rh_change_1h"] = df["relative_humidity"] - df["relative_humidity"].shift(60)
df["rh_change_3h"] = df["relative_humidity"] - df["relative_humidity"].shift(180)
df["temp_change_1h"] = df["air_temp"] - df["air_temp"].shift(60)
df["pressure_change_3h"] = df["air_pressure"] - df["air_pressure"].shift(180)
features = sensors + ["hour", "month", "rh_change_1h", "rh_change_3h",
                      "temp_change_1h", "pressure_change_3h"]

# re-cut the years so they carry the new columns
year1 = df[df["hpwren_timestamp"] < "2012-09-10"]
year2 = df[(df["hpwren_timestamp"] >= "2012-09-10") & (df["hpwren_timestamp"] < "2013-09-10")]
year3 = df[df["hpwren_timestamp"] >= "2013-09-10"]

Positive shifts only: **every feature looks backward.** The first rows have no history, so they are blank; XGBoost accepts blanks.

In [ ]:
# ▶ DEMO -- the early-stopping cell again, on `features`
model = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                      early_stopping_rounds=20, random_state=42)
model.fit(year1[features], year1["dry_3h"],
          eval_set=[(year2[features], year2["dry_3h"])], verbose=False)

report("same as now", year3["dry_now"], year3["dry_3h"])
report("xgboost, + features", model.predict(year3[features]), year3["dry_3h"])

**A fair comparison:** make the model miss exactly as many dry minutes as *same as now*, then count false alarms. We lower the 0.5 threshold until the misses match. This uses year 3's answers, so it compares the two; you could not set it in advance.

In [ ]:
# ▶ DEMO -- equal misses, then compare false alarms
prob = model.predict_proba(year3[features])[:, 1]
baseline_missed = ((year3["dry_now"] == 0) & (year3["dry_3h"] == 1)).sum()
prob_on_dry = prob[year3["dry_3h"] == 1]
threshold = np.quantile(prob_on_dry, baseline_missed / len(prob_on_dry))
print("threshold", round(threshold, 3))
report("same as now", year3["dry_now"], year3["dry_3h"])
report("xgboost, equal misses", (prob >= threshold).astype(int), year3["dry_3h"])

In [ ]:
# ▶ DEMO -- a threshold you could actually use: choose it on year 2, apply it untouched to year 3
prob2 = model.predict_proba(year2[features])[:, 1]
missed2 = ((year2["dry_now"] == 0) & (year2["dry_3h"] == 1)).sum()
threshold2 = np.quantile(prob2[year2["dry_3h"] == 1], missed2 / year2["dry_3h"].sum())
print("threshold from year 2:", round(threshold2, 3))
report("xgboost, year-2 threshold", (prob >= threshold2).astype(int), year3["dry_3h"])

A real improvement, and smaller than 0.958 promised. It took an honest split, a real baseline and new features to find that out. **Whether the crew should use it depends on what a miss costs, and that is not in the data.**

# One More Trap

*Slides: Gradient Boosting · section 8.8*

## 8.8 Is It Raining This Minute?

A rare target, predicted from the other sensors (not a forecast). Only 0.6% of minutes have rain.

In [ ]:
# ▶ DEMO -- `scale_pos_weight=150` counts each rainy minute 150 times in training
no_rain_cols = [c for c in sensors if not c.startswith("rain")]
rain1 = (year1["rain_accumulation"] > 0).astype(int)
rain3 = (year3["rain_accumulation"] > 0).astype(int)
print("rain minutes in year 3:", rain3.sum())

def rain_report(name, pred):
    print(f"{name:<32} accuracy {(pred == rain3).mean():.3f}   caught {((pred == 1) & (rain3 == 1)).sum():>4}"
          f"   false alarms {((pred == 1) & (rain3 == 0)).sum():>6,}")

rain_report('always "no"', np.zeros(len(rain3), dtype=int))
for w in [1, 150]:
    m = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                      scale_pos_weight=w, random_state=42).fit(year1[no_rain_cols], rain1)
    rain_report(f"xgboost, scale_pos_weight={w}", m.predict(year3[no_rain_cols]))

Weighting catches about nine times more rain and raises false alarms about thirty times. **Accuracy falls as the model gets more useful.** At this ratio accuracy is useless: report what was caught and what it cost.

# Your Turn

*Slides: Gradient Boosting · section 8.9*

## 8.9 Will It Be Dry Six Hours From Now?

You may use an agent. You will be asked to explain any cell.

In [ ]:
# ✍️ YOU TRY 8.9
# 1. Build dry_6h. One row is one minute: how many rows ahead is 6 hours?
df["dry_6h"] = (df["relative_humidity"].shift(-0) < 25).astype(int)    # TODO: replace 0
d6 = df.iloc[:-360].copy()

# 2. Split by year (same dates as above)
y1 = d6[d6["hpwren_timestamp"] < "2012-09-10"]
y2 = d6[(d6["hpwren_timestamp"] >= "2012-09-10") & (d6["hpwren_timestamp"] < "2013-09-10")]
y3 = d6[d6["hpwren_timestamp"] >= "2013-09-10"]

# 3. Two baselines on year 3
always_no   = None    # TODO: accuracy of always "no"
same_as_now = None    # TODO: accuracy of dry_now as the prediction

# 4. XGBoost with early stopping on year 2
my_features = features + []    # TODO 5: add ONE column you built that looks only backward
model6 = None                  # TODO: fit like the 8.6 cell, on y1, stopping on y2
my_accuracy = None             # TODO: model6's accuracy on y3

In [ ]:
# ✅ CHECKPOINT 8.9
assert (d6["dry_6h"] != d6["dry_now"]).mean() > 0.05, "dry_6h is (almost) dry_now: shift by 6 HOURS of rows, not 6 rows"
assert (d6["dry_6h"] != d6["dry_3h"]).mean() > 0.02, "dry_6h equals dry_3h: that is 3 hours, not 6"
assert abs(d6["dry_6h"].mean() - 0.26) < 0.02, "about 26% of minutes should be followed by dry air"
assert always_no is not None and same_as_now is not None, "fill in both baselines"
assert model6 is not None and model6.best_iteration < 2999, "fit with early stopping on y2"
assert len(my_features) > len(features), "add one backward-looking column"
print(f'always "no" {always_no:.3f}   same as now {same_as_now:.3f}   your model {my_accuracy:.3f}')
report("same as now", y3["dry_now"], y3["dry_6h"])
report("your model", model6.predict(y3[my_features]), y3["dry_6h"])
print("checkpoint passed")

### ✍️ Two sentences (the graded part)

**Would you give this model to the controlled-burn crew?**
One sentence with the evidence: your model against the better baseline, on year 3, in misses and false alarms. One on what would make you change your answer.

*Your answer:*

# The Same Loop Everywhere

*Slides: Gradient Boosting · section 8.10*

## 8.10 Boosting Is Gradient Descent

Each tree is one step that lowers the loss. XGBoost keeps a score for every tree it adds; plot it and you see the model walking down the loss curve. A neural net does the same walk, nudging its weights instead of adding trees.

In [ ]:
# ▶ DEMO -- watch the loss fall, tree by tree
sample = year1.sample(100000, random_state=0)
walker = XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=3, eval_metric="logloss", random_state=42)
walker.fit(sample[features], sample["dry_3h"],
           eval_set=[(sample[features], sample["dry_3h"]), (year2[features], year2["dry_3h"])], verbose=False)

loss = walker.evals_result()
plt.plot(loss["validation_0"]["logloss"], label="year 1 (training)")
plt.plot(loss["validation_1"]["logloss"], label="year 2 (validation)")
plt.xlabel("trees added"); plt.ylabel("log loss"); plt.title("every tree is one step down"); plt.legend(); plt.show()

In [ ]:
# ▶ DEMO -- gradient descent by hand, on a loss with two valleys
def loss(w):     return 0.15 * w**4 - 1.2 * w**2 + 0.5 * w + 3
def gradient(w): return 0.6 * w**3 - 2.4 * w + 0.5

for start in [-2.9, 2.9]:
    w = start
    for step in range(40):
        w = w - 0.05 * gradient(w)       # 0.05 is the learning rate: try 0.4
    print(f"start at {start:>5}:  end at w = {w:.2f},  loss {loss(w):.2f}")

Same loop, two answers: on a loss that is not a bowl, **where you start decides where you end.** Try a learning rate of 0.4 and watch it overshoot. Neural nets live with this every day; the slides show our team's 99M-parameter GPT walking down its loss for 45,000 steps, and a schedule (AdaQuench) that cuts the learning rate whenever validation stalls: early stopping, applied to the step size.